# Pilot review notebook

Inspect per-video records from `data/dataset` without editing code.
Set `DATASET_ROOT`, run all cells, then use the slider to move between videos.


In [ ]:
import json
from pathlib import Path
import pandas as pd

DATASET_ROOT = Path('data/dataset')
index = pd.read_parquet(DATASET_ROOT / 'index.parquet')
index


In [ ]:
from ipywidgets import interact, IntSlider
from IPython.display import Video, display

def show_record(i=0):
    row = index.iloc[i]
    rec = DATASET_ROOT / 'records' / str(row['video_id'])
    print(f"{row['video_id']}  status={row['status']}  views={row.get('views')}  "
          f"cost=${row['total_usage_cost_usd']:.4f}  latency={row['total_latency_seconds']:.2f}s")
    video = rec / 'source' / 'video.mp4'
    if video.exists():
        display(Video(str(video), width=360))
    for name, path in [
        ('metadata.normalized.json', rec / 'source' / 'metadata.normalized.json'),
        ('shots.json', rec / 'perception' / 'shots.json'),
        ('creative_ir.json', rec / 'decompilation' / 'creative_ir.json'),
        ('canonical_ir.json', rec / 'decompilation' / 'canonical_ir.json'),
        ('validation.json', rec / 'decompilation' / 'validation.json'),
    ]:
        if path.exists():
            print(f'--- {name} ---')
            print(json.dumps(json.loads(path.read_text()), indent=2)[:3000])
    manifest = rec / 'record_manifest.json'
    if manifest.exists():
        m = json.loads(manifest.read_text())
        prov = {k: m[k] for k in ('pipeline_version','prompt_versions','model_ids',
                                  'schema_versions','total_usage_cost_usd','total_latency_seconds') if k in m}
        print('--- provenance/cost ---')
        print(json.dumps(prov, indent=2))
    print('--- evaluation ---')
    evalp = rec / 'review.json'
    print(evalp.read_text() if evalp.exists() else '{"scores": {}, "comments": ""}')

interact(show_record, i=IntSlider(min=0, max=max(len(index)-1, 0), step=1, value=0))
